# Notebook 4 — Flood-risk map & motorist alerts

This is where it all comes together. We combine the risk-factor layers into a single **flood-risk score** for every pixel, check that the score actually agrees with the real 2023 flood, rank the roads by risk, and produce the final maps.

**Method — weighted overlay.** We give each factor a 0–1 risk score (1 = most dangerous), then blend them with weights that reflect how much each one drives flooding:

```
risk = 0.40·(low elevation) + 0.40·(near river) + 0.15·(flat slope) + 0.05·(built-up)
```

This is a transparent, no-training-data approach (a published technique called *multi-criteria flood susceptibility*). We then **validate** it against the SAR flood mask — an independent check that the score predicts reality.

## Step 1 — Load every layer

All rasters share the same grid (we made sure of that in Notebook 3), so they stack like transparent sheets.

In [1]:
import numpy as np
import rasterio
from rasterio.transform import rowcol
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, ListedColormap, BoundaryNorm
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "data" / "outputs"

def load(name):
    with rasterio.open(OUT / name) as src:
        return src.read(1).astype("float32"), src.profile, src.transform

flood, prof, transform = load("flood_mask.tif")
dem, _, _ = load("dem.tif")
slope, _, _ = load("slope.tif")
dist, _, _ = load("dist_river.tif")
built, _, _ = load("builtup.tif")
rivers = gpd.read_file(OUT / "rivers.geojson")
roads = gpd.read_file(OUT / "roads.geojson")

# OSM sometimes stores 'highway'/'name' as lists (e.g. ['Sham Nath Marg']).
# Flatten them to plain strings so they save and serialise cleanly later.
def flatten(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return ", ".join(str(x) for x in v) if len(v) else ""
    return "" if v is None else str(v)

for col in ["highway", "name"]:
    if col in roads.columns:
        roads[col] = roads[col].apply(flatten)

valid = (flood != 255) & np.isfinite(dem)
W, S, E, N = 77.18, 28.50, 77.38, 28.78
print("Layers loaded. Grid:", flood.shape, "| valid pixels:", int(valid.sum()))

Layers loaded. Grid: (3118, 2228) | valid pixels: 6946904


## Step 2 — Turn each factor into a 0–1 risk score

The layers are in different units (metres, degrees, km). We **normalise** each to a 0–1 scale so they're comparable, and flip the ones where *small = dangerous*:

- **Elevation:** low ground is dangerous → invert
- **Slope:** flat is dangerous → invert
- **Distance to river:** close is dangerous → invert
- **Built-up:** already 0/1, kept as is

We clip to the 2nd–98th percentile first so a few extreme pixels don't squash the scale.

In [2]:
def to_score(arr, invert=False):
    lo, hi = np.nanpercentile(arr[valid], 2), np.nanpercentile(arr[valid], 98)
    x = np.clip((arr - lo) / (hi - lo), 0, 1)
    return 1 - x if invert else x

elev_s = to_score(dem, invert=True)
slope_s = to_score(slope, invert=True)
dist_s = to_score(dist, invert=True)
built_s = np.clip(built, 0, 1)
print("Four risk-factor scores ready (each 0–1, where 1 = most flood-prone).")

Four risk-factor scores ready (each 0–1, where 1 = most flood-prone).


## Step 3 — Blend into the flood-risk surface

The weighted sum. Elevation and distance-to-river get the most weight because they drive riverine flooding most. Built-up gets only a small weight — see the validation note below for the honest reason.

In [3]:
W_ELEV, W_DIST, W_SLOPE, W_BUILT = 0.40, 0.40, 0.15, 0.05

raw = W_ELEV * elev_s + W_DIST * dist_s + W_SLOPE * slope_s + W_BUILT * built_s

# Convert to RELATIVE risk: each pixel's percentile rank within the AOI (0-1).
# This is rank-preserving (so it does NOT change the validation AUC below), but it
# spreads the values evenly so the map shows a full low -> high gradient instead of
# being uniformly red across the floodplain corridor.
risk = np.full(raw.shape, np.nan, dtype="float32")
flat = raw[valid]
order = flat.argsort()
pct = np.empty(len(order), dtype="float32")
pct[order] = np.linspace(0, 1, len(order), dtype="float32")
risk[valid] = pct

sprof = prof.copy()
sprof.update(dtype="float32", nodata=np.nan, compress="lzw")
with rasterio.open(OUT / "flood_susceptibility.tif", "w", **sprof) as dst:
    dst.write(risk, 1)
print("Saved flood_susceptibility.tif (relative 0-1 risk).")

Saved flood_susceptibility.tif (relative 0-1 risk).


## Step 4 — Validate against the real 2023 flood

A risk map is only worth trusting if it agrees with reality. We use **AUC** (Area Under the ROC Curve): the probability that a randomly chosen *flooded* pixel scores higher than a randomly chosen *dry* one. 0.5 = useless coin-flip, 1.0 = perfect. Above ~0.8 is strong for a simple model.

We also print each factor's solo AUC — which is where we learn that **built-up actually scores below 0.5**: in Delhi the built-up areas sit on higher ground, so for *this riverine* flood concrete anti-correlates. (It would help for *pluvial* street-waterlogging — a future version.)

In [4]:
def auc(score):
    f = score[valid & (flood == 1)]
    nf = score[valid & (flood == 0)]
    rng = np.random.default_rng(0)
    f = rng.choice(f, min(50000, len(f)), replace=False)
    nf = rng.choice(nf, min(50000, len(nf)), replace=False)
    a = np.concatenate([f, nf])
    ranks = a.argsort().argsort() + 1
    return (ranks[:len(f)].sum() - len(f) * (len(f) + 1) / 2) / (len(f) * len(nf))

print("Single-factor AUCs:")
for nm, s in [("elevation", elev_s), ("slope", slope_s),
              ("distance", dist_s), ("built-up", built_s)]:
    print(f"   {nm:10} {auc(s):.3f}")
print(f"\nCOMBINED risk model AUC: {auc(np.nan_to_num(risk)):.3f}")

Single-factor AUCs:
   elevation  0.735
   slope      0.778
   distance   0.850
   built-up   0.305

COMBINED risk model AUC: 0.841


## Step 5 — Classify into four risk zones

Continuous scores are precise but hard to read on a map, so we bucket them into four familiar zones.

In [5]:
zones = np.full(risk.shape, 255, dtype="uint8")
zones[valid] = 1                # Low
zones[risk >= 0.25] = 2         # Medium
zones[risk >= 0.50] = 3         # High
zones[risk >= 0.75] = 4         # Very High
zones[~valid] = 255

zprof = prof.copy(); zprof.update(dtype="uint8", nodata=255, compress="lzw")
with rasterio.open(OUT / "risk_zones.tif", "w", **zprof) as dst:
    dst.write(zones, 1)

px_km2 = (abs(transform.a) * 111.32 * np.cos(np.deg2rad((S + N) / 2))) * (abs(transform.e) * 111.32)
for v, label in [(1, "Low"), (2, "Medium"), (3, "High"), (4, "Very High")]:
    print(f"   {label:10}: {(zones == v).sum() * px_km2:6.1f} km²")

   Low       :  152.4 km²
   Medium    :  152.4 km²
   High      :  152.4 km²
   Very High :  152.4 km²


## Step 6 — Rank the roads (the motorist alert)

For each major road we sample the risk surface along its length and take the average. The highest-scoring roads are the ones most worth warning drivers about. We save the scored roads and a CSV of the worst offenders.

In [6]:
risk_filled = np.nan_to_num(risk, nan=0.0)

def road_risk(geom, n=8):
    pts = [geom.interpolate(d, normalized=True) for d in np.linspace(0, 1, n)]
    xs = [p.x for p in pts]; ys = [p.y for p in pts]
    rows, cols = rowcol(transform, xs, ys)
    rows = np.clip(np.array(rows), 0, risk.shape[0] - 1)
    cols = np.clip(np.array(cols), 0, risk.shape[1] - 1)
    return float(np.mean(risk_filled[rows, cols]))

roads = roads[roads.geom_type == "LineString"].copy()
roads["risk"] = roads.geometry.apply(road_risk)
roads.to_file(OUT / "roads_risk.geojson", driver="GeoJSON")

named = roads[roads["name"] != ""]
top = (named.groupby("name")["risk"].mean()
            .sort_values(ascending=False).head(15))
top.round(3).to_csv(OUT / "top_risk_roads.csv", header=["mean_risk"])
print("Top 10 highest-risk named roads (for motorist alerts):")
print(top.head(10).round(3).to_string())

Top 10 highest-risk named roads (for motorist alerts):
name
Jhasi Rani Lakshmibai Road                         0.956
DND KMP Expressway                                 0.953
Bandh Road                                         0.948
Baba Banda Singh Bahadur Setu Ramp                 0.939
Vishwakarma Flyover, Jhasi Rani Lakshmibai Road    0.933
Dadri Main Road                                    0.929
Aurangzeb Road                                     0.928
Udyog Marg                                         0.927
Maharaja Agrasen Marg Underpass                    0.925
Kalindi Kunj Bridge                                0.924


## Step 7 — The final risk map (the hero image)

We compose the publication map: a **hillshade** (simulated terrain shading from the DEM) as a neutral backdrop, the flood-risk surface in warm colours on top, the river in blue, and the major roads. This PNG goes in the README.

In [7]:
px_x = abs(transform.a) * 111_320 * np.cos(np.deg2rad((S + N) / 2))
px_y = abs(transform.e) * 111_320
ls = LightSource(azdeg=315, altdeg=45)
hill = ls.hillshade(np.nan_to_num(dem, nan=float(np.nanmean(dem))), vert_exag=5, dx=px_x, dy=px_y)
extent = [W, E, S, N]

fig, ax = plt.subplots(figsize=(11, 13))
ax.imshow(hill, cmap="gray", extent=extent, alpha=0.6)
rim = ax.imshow(np.ma.masked_invalid(risk), cmap="YlOrRd", alpha=0.65,
                extent=extent, vmin=0, vmax=1)
rivers.plot(ax=ax, color="#1f5fa8", linewidth=1.4)
roads.plot(ax=ax, color="black", linewidth=0.3, alpha=0.5)

ax.set_xlim(W, E); ax.set_ylim(S, N)   # crop to the data box
cbar = fig.colorbar(rim, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("Relative flood risk (0 = low, 1 = very high)")
ax.set_title("Yamuna Urban Flood Risk — Delhi\n"
             "Weighted-overlay model, validated against Sentinel-1 SAR (July 2023 flood)",
             fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(OUT / "yamuna_flood_risk_2023.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved yamuna_flood_risk_2023.png")

Saved yamuna_flood_risk_2023.png


/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_14690/1820442495.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8 — The observed flood-extent map (July 2023)

A companion map showing the actual SAR-detected flood, for the README's "what happened" panel.

In [8]:
fig, ax = plt.subplots(figsize=(9, 11))
ax.imshow(hill, cmap="gray", extent=extent, alpha=0.6)
flood_only = np.ma.masked_where(flood != 1, flood)
ax.imshow(flood_only, cmap="cool", extent=extent, alpha=0.8)
rivers.plot(ax=ax, color="#1f5fa8", linewidth=1.4)
ax.set_xlim(W, E); ax.set_ylim(S, N)   # crop to the data box
ax.set_title("Observed flood extent — Sentinel-1 SAR change detection\nYamuna, Delhi · July 2023",
             fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(OUT / "yamuna_flood_extent_2023.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved yamuna_flood_extent_2023.png")

Saved yamuna_flood_extent_2023.png


/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_14690/1119573324.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 9 — Interactive map (bonus)

An HTML map (OpenStreetMap basemap) with the risk zones overlaid and the top-risk roads marked — pannable/zoomable, and embeddable later. Open `data/outputs/yamuna_flood_risk_interactive.html` in a browser.

In [9]:
import folium, branca.colormap as cm

m = folium.Map(location=[(S + N) / 2, (W + E) / 2], zoom_start=12, tiles="OpenStreetMap")
cmap = cm.LinearColormap(["#2c7bb6", "#ffffbf", "#d7191c"], vmin=0, vmax=1,
                         caption="Flood risk")

def style(feat):
    r = feat["properties"]["risk"]
    return {"color": cmap(r), "weight": 2.5 + 3 * r}

folium.GeoJson(roads_to := roads[["name", "highway", "risk", "geometry"]].fillna({"name": ""}),
               style_function=style,
               tooltip=folium.GeoJsonTooltip(fields=["name", "highway", "risk"])).add_to(m)
cmap.add_to(m)
m.save(OUT / "yamuna_flood_risk_interactive.html")
print("Saved yamuna_flood_risk_interactive.html")
m

Saved yamuna_flood_risk_interactive.html


## Done!

You built a complete urban flood pipeline: SAR flood detection → context layers → a validated flood-risk model → ranked roads → publication maps. Final outputs in `data/outputs/`:

- `flood_susceptibility.tif` — continuous 0–1 risk
- `risk_zones.tif` — four risk classes
- `top_risk_roads.csv` — the motorist watch-list
- `yamuna_flood_risk_2023.png` — the hero map
- `yamuna_flood_extent_2023.png` — observed flood
- `yamuna_flood_risk_interactive.html` — interactive map

**Next:** push everything to GitHub with a README that tells the story.